Import and definitions

In [1]:
import pandas as pd
import numpy as np


# Configuration
INPUT_FILE = "../data/raw/recipes_images.json"
OUTPUT_FILE = "../data/processed/recipes_clean_all_final.json"

ALLOWED_MEALS = ["Breakfast", "Lunch", "Dinner"]
MIN_INGREDIENTS = 3
MAX_INGREDIENTS = 15
MIN_INSTRUCTION_LENGTH = 150

SIMPLE_COOKING_TAGS = [
    "30 Minutes or Less", "5 Ingredients or Fewer", "Basically", "Budget Cooking", 
    "Easy", "Make Ahead", "Meal Prep", "One-Pot Meals", "Quick", 
    "Sheet-Pan Dinners", "Weeknight Meals"
]
'''
more to exclude
    "Dip", "Fritter", "Guacamole", "Hummus", "Kimchi", "Nachos", "Pickle", "Condiment", 
    "Marinade", "Salad Dressing", "Salsa", "Sauce", "Spreads",
'''

# excluded lists 
EXCLUDED_TITLE_WORDS = [
    "Alcohol", "Beer", "Beverage", "Bourbon", "Brandy", "Brownie", "Cake", "Candy", "Caramel", "Champagne", "Cheesecake",
    "Chocolaty", "Cider", "Cocktail", "Coffee", "Cookie", "Cordial", "Crisp", "Crumble", "Cupcake", "Custard", "Daiquiri",
    "Dessert", "Donut", "Doughnut", "Drink", "Fizz", "Flan", "Frosting", "Fudge", "Gelato", "Gin", "Ice Cream", "Jam", "Jelly",
    "Juice", "Liqueur", "Liquor", "Margarita", "Martini", "Milkshake", "Mojito", "Mousse", "Negroni", "Panna Cotta", "Pastries", "Pastry",
    "Pie", "Popsicle", "Pudding", "Punch", "Rum", "Sake", "Sangria", "Scotch", "Sherry", "Smoothie", "Soda", "Sorbet",
    "Spritz", "Sundae", "Sweet", "Syrup", "Tart", "Tea", "Tequila", "Toffee", "Torte", "Vodka", "Whiskey", "Wine", 
    "Dip", "Fritter", "Guacamole", "Hummus", "Kimchi", "Nachos", "Pickle", "Condiment", "Marinade", "Salad Dressing", "Salsa", "Sauce", "Spreads",
]

EXCLUDED_TYPE_WORDS = [
    "Absinthe", "Agua Fresca", "Alcohol", "Amaretto", "Amaro", "Aperitif", "Aperol", "Apple Cider", "Aquavit", "Armagnac",
    "Bar Cookie", "Beer", "Beverages", "Biscotti", "Bitters", "Black Tea", "Bloody Mary", "Bourbon", "Brandy", "Bread Pudding",
    "Brownie", "Cake", "Calvados", "Campari", "Candy", "Cava", "Champagne", "Chartreuse", "Cheesecake", "Chamomile Tea",
    "Cinnamon Roll", "Cobbler", "Cocktail", "Coffee", "Cognac", "Cointreau", "Cold Drink", "Compote", "Cookie", "Crisp",
    "Crumble", "Crème de Cassis", "Cupcake", "Custard", "Cynar", "Daiquiri", "Dessert", "Doughnut", "Earl Grey", "Espresso",
    "Flan", "Frangelico", "Frosting", "Frozen Dessert", "Fruit Dessert", "Gelato", "Gin", "Grand Marnier", "Granita", "Grappa",
    "Green Tea", "Grenadine", "Hard Cider", "Hot Chocolate", "Hot Drink", "Ice Cream", "Iced Coffee", "Iced Tea", "Jam",
    "Kahlua", "Kirsch", "Layer Cake", "Lemonade", "Lillet", "Liqueur", "Loaf Cake", "Margarita", "Marsala", "Martini",
    "Mezcal", "Milkshake", "Mojito", "Mousse", "Negroni", "Nonalcoholic", "Panna Cotta", "Parfait", "Pastries", "Pastry",
    "Pernod Pastis", "Pie", "Pimm's", "Pisco", "Pitcher Drink", "Popsicle", "Port", "Prosecco", "Pudding", "Punch",
    "Red Wine", "Rice Pudding", "Rosé", "Rum", "Rye Whiskey", "Sake", "Sangria", "Scone", "Scotch", "Sherry", "Smoothie",
    "Sorbet", "Sparkling Wine", "Spritz", "St-Germain", "Sundae", "Suze", "Tart", "Tea", "Tequila", "Torte", "Triple Sec",
    "Turnover", "Vermouth", "Vodka", "Whiskey", "White Wine", "Wine", "Dip", "Fritter", "Guacamole", "Hummus", "Kimchi", "Nachos", 
    "Pickle", "Condiment", "Marinade", "Salad Dressing", "Salsa", "Sauce", "Spreads",
]


Load data:

In [2]:
with open(INPUT_FILE, "r", encoding="utf-8") as file:
    df_raw_recipes = pd.read_json(file)
print(f"Loaded {len(df_raw_recipes)} recipes.")

# df_raw.head()

Loaded 20935 recipes.


In [3]:
top_level_columns = df_raw_recipes.columns.tolist()
print(f"Top-level columns: {top_level_columns}")

Top-level columns: ['title', 'description', 'ingredients', 'instructions', 'cooking_time', 'servings', 'ratings', 'tags', 'publish_date', 'image_filename']


Delete unnecessary attributes:

In [4]:
# 1. Drop top-level 
cols_to_drop = ['ratings', 'servings', 'publish_date'] 
df_raw = df_raw_recipes.drop(columns=cols_to_drop, errors='ignore')

# 2. Clean the nested tags column

tags_to_remove = ['technique', 'equipment', 'source', 'cne-video-tags', 'occasion']

def clean_tags_dict(tags):
    if isinstance(tags, dict):
        # excluding the unwanted keys
        return {k: v for k, v in tags.items() if k not in tags_to_remove}
    return tags

df_raw['tags'] = df_raw_recipes['tags'].apply(clean_tags_dict)

print("Dropped top-level columns")
print(f"Removed keys from tags: {', '.join(tags_to_remove)}")
# Verify the tags dictionary of the first row
print("\ncleaned tags:")
print(df_raw['tags'].iloc[0].keys())

Dropped top-level columns
Removed keys from tags: technique, equipment, source, cne-video-tags, occasion

cleaned tags:
dict_keys(['type', 'cuisine', 'ingredient', 'meal', 'special-consideration', 'simple-cooking'])


Define mandatory columns:

In [5]:
#  Filter out recipes missing basic structure (Title, Ingredients, Instructions, Image)

#  Normalize instructions into a single string 
def flatten_instructions(instr):
    if isinstance(instr, dict): return " ".join(str(v) for v in instr.values())
    if isinstance(instr, list): return " ".join(str(s) for s in instr)
    return str(instr or "")

df_step1 = df_raw.copy()
df_step1['instructions_flat'] = df_step1['instructions'].apply(flatten_instructions)

# Filter out rows with missing mandatory data
df_step1 = df_step1[
    (df_step1['title'].str.strip() != "") & 
    (df_step1['ingredients'].notna()) & 
    (df_step1['instructions_flat'].str.strip() != "") &
    (df_step1['image_filename'].notna())
]
print(f"After basic check: {len(df_step1)}")

After basic check: 9475


Filter: allowed meals

In [6]:
# Extract tags and identify the meal type
# check which of the ALLOWED_MEALS appears in the 'meal' tag list
df_step2 = df_step1.copy()

def get_primary_meal(tags):
    meal_tags = (tags or {}).get('meal', [])
    for meal in ALLOWED_MEALS:
        if meal in meal_tags:
            return meal
    return None

df_step2['primary_meal'] = df_step2['tags'].apply(get_primary_meal)

# Remove recipes that don't fall into allowed meal categories
df_step2 = df_step2.dropna(subset=['primary_meal'])
print(f"After meal filtering: {len(df_step2)}")

After meal filtering: 7199


In [7]:
top_level_columns = df_step2.columns.tolist()
print(f"Top-level columns: {top_level_columns}")

all_tag_keys = set()
for tag_dict in df_step2['tags'].dropna():
    if isinstance(tag_dict, dict):
        all_tag_keys.update(tag_dict.keys())

print("\n Attributes inside tags:")
print(sorted(list(all_tag_keys)))



Top-level columns: ['title', 'description', 'ingredients', 'instructions', 'cooking_time', 'tags', 'image_filename', 'instructions_flat', 'primary_meal']

 Attributes inside tags:
['cuisine', 'ingredient', 'meal', 'simple-cooking', 'special-consideration', 'type']


In [8]:
import pandas as pd

# 1. Get the unique values (array)
unique_titles = df_step2['title'].unique()

# 2. Convert to a Series and  save 
pd.Series(unique_titles, name='title').to_csv("../data/processed/meal_titles.csv", index=False)

print("Saved successfully to CSV")

Saved successfully to CSV


Remove drinks, alcohol and desserts like cakes, ice-cream etc.

In [8]:
# exclusion sets 
excl_titles = set(w.lower() for w in EXCLUDED_TITLE_WORDS)
excl_types = set(w.lower() for w in EXCLUDED_TYPE_WORDS)

def should_keep(row):
    # Check Title
    # to find exact matches
    
    title_words = row['title'].lower().replace('-', ' ').split()
    if any(w in excl_titles for w in title_words):
        return False
    
    # Check Tags 
    # Check if any tag in the 'type' list exists in our exclusion set
    row_types = (row['tags'] or {}).get('type', [])
    if any(t.lower() in excl_types for t in row_types):
        return False
        
    return True

# Apply the filter
df_step3 = df_step2[df_step2.apply(should_keep, axis=1)].copy()

print(f"Remaining: {len(df_step3)}")

Remaining: 5495


Ingredient Count Filter

In [9]:
# Step 4: Count ingredients and filter within range (3-15)
df_step4 = df_step3.copy()
df_step4['ing_count'] = df_step4['ingredients'].apply(lambda x: len(x) if isinstance(x, list) else 0)

df_step4 = df_step4[
    (df_step4['ing_count'] >= MIN_INGREDIENTS) & 
    (df_step4['ing_count'] <= MAX_INGREDIENTS)
]
print(f"After ingredient count filter: {len(df_step4)}")

After ingredient count filter: 4631


Simple cooking for Lunch and dinner +  min instructions length

In [13]:
# for Lunch/Dinner, require  minimum instruction length
df_step5 = df_step4.copy()

df_step5['instr_len'] = df_step5['instructions_flat'].str.len()

# Logic: If Lunch/Dinner, must have enough instructions
mask_lunch_dinner = df_step5['primary_meal'].isin(['Lunch', 'Dinner'])
mask_valid_len = (df_step5['instr_len'] >= MIN_INSTRUCTION_LENGTH)

# Drop rows that are Lunch/Dinner but DON'T meet the length requirement
drop_mask = mask_lunch_dinner & ~mask_valid_len
df_step5 = df_step5[~drop_mask]

print(f"After Lunch/Dinner instruction length check: {len(df_step5)}")

After Lunch/Dinner instruction length check: 4602


In [14]:
# True if the cuisine list is empty
no_cuisine_count = df_step5['tags'].apply(lambda x: len(x.get('cuisine', [])) == 0).sum()

print(f"Number of recipes with no cuisine: {no_cuisine_count}")
print(f"Percentage of total: {(no_cuisine_count / len(df_step5)) * 100:.2f}%")

Number of recipes with no cuisine: 2793
Percentage of total: 60.69%


In [15]:
# 1.True if it has NO cuisine, False if it has cuisine
has_no_cuisine = df_step5['tags'].apply(lambda x: len(x.get('cuisine', [])) == 0)

df_step6 = df_step5[~has_no_cuisine].copy()

print(f"Original count: {len(df_step5)}")
print(f"Recipes removed: {has_no_cuisine.sum()}")
print(f"Remaining after removing no-cuisine recipes: {len(df_step6)}")

Original count: 4602
Recipes removed: 2793
Remaining after removing no-cuisine recipes: 1809


In [16]:
df_step6.to_json("../data/processed/recipes_final_dataset2.json", 
                  orient='records', 
                  indent=2, 
                  force_ascii=False)

In [17]:
with open("../data/processed/recipes_final_dataset2.json", "r", encoding="utf-8") as file:
    df_recipes_final = pd.read_json(file)
print(f"Loaded {len(df_recipes_final)} recipes.")


Loaded 1809 recipes.


In [18]:

meal_counts = df_recipes_final['primary_meal'].value_counts()

print(meal_counts)

primary_meal
Dinner       1115
Lunch         579
Breakfast     115
Name: count, dtype: int64


In [19]:
# top-level column names
top_level_columns = df_recipes_final.columns.tolist()

#  unique keys inside the "tags"column
all_tag_keys = set()
for tag_dict in df_recipes_final['tags'].dropna():
    if isinstance(tag_dict, dict):
        all_tag_keys.update(tag_dict.keys())

print("Top Level:")
print(top_level_columns)

print("\n Attributes inside tags:")
print(sorted(list(all_tag_keys)))

Top Level:
['title', 'description', 'ingredients', 'instructions', 'cooking_time', 'tags', 'image_filename', 'instructions_flat', 'primary_meal', 'ing_count', 'instr_len']

 Attributes inside tags:
['cuisine', 'ingredient', 'meal', 'simple-cooking', 'special-consideration', 'type']


In [20]:
tags_to_check = ['cuisine', 'meal', 'simple-cooking', 'special-consideration']

for tag_name in tags_to_check:
    
    unique_values = df_raw['tags'].apply(lambda x: x.get(tag_name, []) if isinstance(x, dict) else []) \
                                 .explode() \
                                 .dropna() \
                                 .unique()
    
    print(f"\n Unique {tag_name.upper()} ({len(unique_values)} found)")
    print(sorted(list(unique_values)))


 Unique CUISINE (107 found)
['African', 'American', 'Argentinean', 'Armenian', 'Asian', 'Australian', 'Austrian', 'Bangladeshi', 'Basque', 'Belgian', 'Brazilian', 'British', 'Burmese', 'Cajun & Creole', 'California Cuisine', 'Canadian', 'Cantonese', 'Caribbean', 'Central American', 'Chinese', 'Chinese-American', 'Colombian', 'Cuban', 'Danish', 'Dominican', 'East African', 'East Asian', 'Eastern European', 'Egyptian', 'English', 'Ethiopian', 'European', 'Filipino', 'French', 'Georgian', 'German', 'Greek', 'Haitian', 'Hawaiian', 'Hungarian', 'Indian', 'Indonesian', 'Iranian', 'Irish', 'Israeli', 'Italian', 'Italian American', 'Jamaican', 'Japanese', 'Jewish', 'Korean', 'Laotian', 'Latin American', 'Lebanese', 'Levantine', 'Low Country Cuisine', 'Malaysian', 'Mediterranean', 'Mexican', 'Middle Eastern', 'Moroccan', 'Native American', 'New England', 'New Zealand', 'Nigerian', 'North African', 'Norwegian', 'Oaxacan', 'Pakistani', 'Palestinian', 'Persian', 'Peruvian', 'Polish', 'Portuguese'

The code under this markdown is for tests only do not run if not needed 

In [ ]:
subset_df = pd.DataFrame()

subset_df['title'] = df_step7['title']

subset_df['cuisine'] = df_step7['tags'].apply(lambda x: x.get('cuisine', []) if isinstance(x, dict) else [])
subset_df['meal_tags'] = df_step7['primary_meal']

subset_df.to_json("../data/processed/recipes_simple_subset.json", 
                  orient='records', 
                  indent=2, 
                  force_ascii=False)

# View the result
subset_df.head()

In [ ]:
subset_df.to_json("../data/processed/recipes_step3.json", 
                orient='records', 
                indent=2, 
                force_ascii=False)

print("Saved successfully to JSON")